In [ ]:
# ==============================================================================
# Block 1: Tokenizer + Model Architecture + Model Loading
# ==============================================================================

import torch
import torch.nn as nn
import numpy as np


if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

WL_FULL = np.arange(200, 801, 1)


class ThinFilmTokenizer:
    """
    Token 0       : <BOS>
    Token 1       : <PAD>
    Token 2~147   : eps = 11.7, thickness = 10~300 nm, step = 2 nm
    Token 148~293 : eps = 3.9,  thickness = 10~300 nm, step = 2 nm
    """

    def __init__(
        self,
        bos_token=0,
        pad_token=1,
        eps_values=(11.7, 3.9),
        thickness_min_nm=10.0,
        thickness_max_nm=300.0,
        thickness_step_nm=2.0,
    ):
        self.bos_token = bos_token
        self.pad_token = pad_token
        self.eps_values = eps_values
        self.thickness_min_nm = thickness_min_nm
        self.thickness_max_nm = thickness_max_nm
        self.thickness_step_nm = thickness_step_nm

        self.num_bins = int(
            round(
                (thickness_max_nm - thickness_min_nm)
                / thickness_step_nm
            )
        ) + 1

        self.vocab_size = 2 + 2 * self.num_bins

    def encode_layer(self, material, thickness):
        if thickness < 5.0:
            return self.pad_token

        thickness = np.clip(
            thickness,
            self.thickness_min_nm,
            self.thickness_max_nm,
        )

        thickness_idx = int(
            np.round(
                (thickness - self.thickness_min_nm)
                / self.thickness_step_nm
            )
        )

        thickness_idx = np.clip(
            thickness_idx,
            0,
            self.num_bins - 1,
        )

        if abs(material - self.eps_values[0]) < abs(material - self.eps_values[1]):
            return 2 + thickness_idx

        return 2 + self.num_bins + thickness_idx

    def decode_token(self, token):
        if token in (self.bos_token, self.pad_token):
            return 0.0, 0.0

        if 2 <= token < 2 + self.num_bins:
            thickness = (
                self.thickness_min_nm
                + (token - 2) * self.thickness_step_nm
            )

            return self.eps_values[0], float(thickness)

        thickness = (
            self.thickness_min_nm
            + (token - 2 - self.num_bins)
            * self.thickness_step_nm
        )

        return self.eps_values[1], float(thickness)


class AutoregressiveTokenDecoder(nn.Module):
    def __init__(
        self,
        input_dim=601,
        num_layers=10,
        vocab_size=294,
        d_model=256,
        nhead=8,
        num_layers_dec=6,
        dim_feedforward=1024,
        dropout=0.05,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.num_layers = num_layers
        self.d_model = d_model

        self.spec_conv = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Conv1d(128, d_model, kernel_size=3, padding=1),
        )

        self.spec_pos_emb = nn.Parameter(
            torch.randn(1, input_dim, d_model) * 0.02
        )

        self.token_emb = nn.Embedding(vocab_size, d_model)

        self.pos_emb = nn.Parameter(
            torch.randn(1, num_layers + 1, d_model) * 0.02
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
        )

        self.transformer_decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_layers_dec,
        )

        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, spectrum, tgt_tokens):
        spec_feats = self.spec_conv(
            spectrum.unsqueeze(1)
        ).transpose(1, 2)

        spec_feats = spec_feats + self.spec_pos_emb

        _, seq_len = tgt_tokens.size()

        tgt_emb = (
            self.token_emb(tgt_tokens)
            + self.pos_emb[:, :seq_len, :]
        )

        causal_mask = nn.Transformer.generate_square_subsequent_mask(
            seq_len
        ).to(spectrum.device)

        out = self.transformer_decoder(
            tgt=tgt_emb,
            memory=spec_feats,
            tgt_mask=causal_mask,
        )

        return self.head(out)


def load_trained_model(
    model_path="thin_film_transformer_variable_final.pth",
    device=DEVICE,
):
    """
    同時支援：
    1. 純 state_dict checkpoint
    2. model_config + tokenizer_config + state_dict 發布版 checkpoint
    """

    checkpoint = torch.load(
        model_path,
        map_location=device,
        weights_only=False,
    )

    default_model_config = {
        "input_dim": 601,
        "num_layers": 10,
        "vocab_size": 294,
        "d_model": 256,
        "nhead": 8,
        "num_layers_dec": 6,
        "dim_feedforward": 1024,
        "dropout": 0.05,
    }

    default_tokenizer_config = {
        "bos_token": 0,
        "pad_token": 1,
        "eps_values": [11.7, 3.9],
        "thickness_min_nm": 10.0,
        "thickness_max_nm": 300.0,
        "thickness_step_nm": 2.0,
    }

    # 發布版 checkpoint
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
        model_config = checkpoint.get("model_config", default_model_config)
        tokenizer_config = checkpoint.get(
            "tokenizer_config",
            default_tokenizer_config,
        )

    # Block 2 直接儲存的純 state_dict
    else:
        state_dict = checkpoint
        model_config = default_model_config
        tokenizer_config = default_tokenizer_config

    model = AutoregressiveTokenDecoder(
        **model_config
    ).to(device)

    model.load_state_dict(state_dict)
    model.eval()

    tokenizer = ThinFilmTokenizer(
        **tokenizer_config
    )

    print(
        f"Model loaded | "
        f"Input Dim: {model_config['input_dim']} | "
        f"Vocab Size: {model_config['vocab_size']} | "
        f"Device: {device}"
    )

    return model, tokenizer, model_config

In [ ]:
# ==============================================================================
# Block 2: TMM Physics Engine + Random Variable-Layer Test Structure
# ==============================================================================

import numpy as np
from simulation import Certain_Wavelength


def calculate_transmittance_spectrum(epsilons, thickness_nm, wavelengths_nm=WL_FULL,
                                     eps_in=1.0, eps_out=1.0):
    """
    使用與 Dataset Generation 相同的 TMM 計算 Transmittance Spectrum。

    epsilons     : 各有效薄膜層的介電常數
    thickness_nm : 各有效薄膜層的厚度 (nm)

    return:
        transmittance spectrum, shape = (601,)
    """

    active_mask = thickness_nm > 0.0

    active_eps = np.asarray(epsilons[active_mask], dtype=np.float32)
    active_thickness_nm = np.asarray(thickness_nm[active_mask], dtype=np.float32)

    params = np.empty(len(active_eps) * 2, dtype=np.float32)
    params[0::2] = active_eps
    params[1::2] = active_thickness_nm * 1e-9

    component = Certain_Wavelength(eps_in, eps_out, params)

    transmittance = []

    for wavelength_nm in wavelengths_nm:
        _, T, _ = component.RTA(wavelength_nm * 1e-9, eps_in, eps_out)
        transmittance.append(T)

    return np.asarray(transmittance, dtype=np.float32)


def generate_random_test_sample(num_layers=None):
    """
    隨機生成全新的 2~10 層薄膜結構。

    結構規則：
        - 層數：2~10
        - 第一層材料隨機為 eps = 3.9 或 11.7
        - 後續材料交替排列
        - 厚度使用 10~300 nm 的連續均勻分布
        - 不足 10 層的位置以 0 nm padding

    Output:
        structure_params : shape = (20,)
        target_spectrum  : shape = (601,)
    """

    max_layers = 10

    if num_layers is None:
        active_layers = np.random.randint(2, max_layers + 1)
    else:
        active_layers = int(np.clip(num_layers, 2, max_layers))

    first_eps = np.random.choice([3.9, 11.7])
    second_eps = 11.7 if first_eps == 3.9 else 3.9

    epsilons = np.zeros(max_layers, dtype=np.float32)
    thickness_nm = np.zeros(max_layers, dtype=np.float32)

    for i in range(active_layers):
        epsilons[i] = first_eps if i % 2 == 0 else second_eps
        thickness_nm[i] = np.random.uniform(10.0, 300.0)

    target_spectrum = calculate_transmittance_spectrum(
        epsilons,
        thickness_nm,
    )

    structure_params = np.empty(max_layers * 2, dtype=np.float32)
    structure_params[0::2] = epsilons
    structure_params[1::2] = thickness_nm

    return structure_params, target_spectrum

In [ ]:
# ==============================================================================
# Block 3: Beam Search Inference for 2~10 Layer Structures
# ==============================================================================

import torch
import numpy as np


def run_beam_search_variable_layers(model, target_spectrum, beam_width=24,
                                    max_layers=10, min_layers=2, device=DEVICE):
    """
    使用 Beam Search 進行 2~10 層薄膜反向設計。

    約束：
        - Token 0 為 BOS，不允許生成
        - 前 min_layers 層不可生成 PAD
        - 一旦生成 PAD，後續只能繼續生成 PAD
        - 相鄰兩層材料必須交替

    最後使用 TMM 計算所有 Beam 候選的實際 Transmittance Spectrum，
    並選擇 Spectrum MAE 最小的結構。

    Output:
        best_eps          : shape = (10,)
        best_thickness_nm : shape = (10,)
        best_spectrum     : shape = (601,)
        best_spectrum_mae : float
    """

    model.eval()

    spectrum_tensor = torch.tensor(
        target_spectrum,
        dtype=torch.float32,
        device=device,
    ).unsqueeze(0)

    sequences = [(0.0, [tokenizer.bos_token])]

    with torch.no_grad():

        for step in range(max_layers):

            all_candidates = []

            for score, sequence in sequences:

                input_tokens = torch.tensor(
                    [sequence],
                    dtype=torch.long,
                    device=device,
                )

                logits = model(spectrum_tensor, input_tokens)
                next_logits = logits[0, -1, :].clone()

                # BOS 只能出現在序列開頭
                next_logits[tokenizer.bos_token] = -float("inf")

                last_token = sequence[-1]

                # 一旦出現 PAD，後續只能繼續 PAD
                if last_token == tokenizer.pad_token:
                    mask = torch.full_like(next_logits, -float("inf"))
                    mask[tokenizer.pad_token] = next_logits[tokenizer.pad_token]
                    next_logits = mask

                else:
                    # 至少必須存在 min_layers 層
                    if step < min_layers:
                        next_logits[tokenizer.pad_token] = -float("inf")

                    # 材料必須交替
                    if 2 <= last_token <= 147:
                        next_logits[2:148] = -float("inf")

                    elif 148 <= last_token <= 293:
                        next_logits[148:294] = -float("inf")

                log_probs = torch.log_softmax(next_logits, dim=-1)

                top_log_probs, top_tokens = torch.topk(
                    log_probs,
                    k=min(beam_width, tokenizer.vocab_size),
                )

                for log_prob, token in zip(top_log_probs, top_tokens):

                    if torch.isneginf(log_prob):
                        continue

                    new_score = score + float(log_prob.item())
                    new_sequence = sequence + [int(token.item())]

                    all_candidates.append(
                        (new_score, new_sequence)
                    )

            all_candidates.sort(
                key=lambda x: x[0],
                reverse=True,
            )

            sequences = all_candidates[:beam_width]


    """
    將 Beam Search 保留下來的候選結構重新送入 TMM，
    最終依 Spectrum MAE 選擇物理上最符合 Target 的結果。
    """

    best_spectrum_mae = float("inf")
    best_eps = None
    best_thickness_nm = None
    best_spectrum = None

    for score, sequence in sequences:

        predicted_tokens = sequence[1:]

        epsilons = []
        thickness_nm = []

        for token in predicted_tokens:
            eps, thickness = tokenizer.decode_token(token)

            epsilons.append(eps)
            thickness_nm.append(thickness)

        epsilons = np.asarray(epsilons, dtype=np.float32)
        thickness_nm = np.asarray(thickness_nm, dtype=np.float32)

        simulated_spectrum = calculate_transmittance_spectrum(
            epsilons,
            thickness_nm,
        )

        spectrum_mae = float(
            np.mean(np.abs(target_spectrum - simulated_spectrum))
        )

        if spectrum_mae < best_spectrum_mae:
            best_spectrum_mae = spectrum_mae
            best_eps = epsilons
            best_thickness_nm = thickness_nm
            best_spectrum = simulated_spectrum

    return (
        best_eps,
        best_thickness_nm,
        best_spectrum,
        best_spectrum_mae,
    )

In [ ]:
# ==============================================================================
# Block 4: L-BFGS-B Continuous Thickness Refinement
# ==============================================================================

import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt


def run_lbfgsb_variable_layers(
    target_params,
    target_spectrum,
    predicted_params,
    max_iter=100,
    plot=True,
):
    """
    以 Beam Search 預測結構作為初始點，
    固定材料種類與層數，只使用 L-BFGS-B 微調有效薄膜層的連續厚度。

    Input:
        target_params    : Ground Truth 結構，shape = (20,)
        target_spectrum  : Target Transmittance Spectrum，shape = (601,)
        predicted_params : Beam Search 預測結構，shape = (20,)

    Output:
        optimized_spectrum_mae  : 最佳化後 Spectrum MAE
        optimized_thickness_mae : 最佳化後 Thickness MAE
        optimized_params        : 最佳化後結構，shape = (20,)
    """

    beam_eps = predicted_params[0::2].copy()
    beam_thickness_nm = predicted_params[1::2].copy()

    # Beam Search 判定存在的有效薄膜層
    active_indices = np.where(beam_thickness_nm > 0.0)[0]
    initial_thickness_nm = beam_thickness_nm[active_indices].astype(np.float64)

    # L-BFGS-B 只改 Thickness，材料保持 Beam Search 的預測
    fixed_eps = beam_eps.copy()

    def loss_function(thickness_nm):
        current_thickness_nm = np.zeros_like(beam_thickness_nm, dtype=np.float64)
        current_thickness_nm[active_indices] = thickness_nm

        simulated_spectrum = calculate_transmittance_spectrum(
            fixed_eps,
            current_thickness_nm,
        )

        return float(
            np.mean((target_spectrum - simulated_spectrum) ** 2)
        )

    bounds = [(10.0, 300.0)] * len(initial_thickness_nm)

    if len(initial_thickness_nm) > 0 and max_iter > 0:
        result = minimize(
            loss_function,
            x0=initial_thickness_nm,
            method="L-BFGS-B",
            bounds=bounds,
            options={
                "eps": 1e-2,
                "maxiter": max_iter,
                "ftol": 1e-12,
                "gtol": 1e-7,
            },
        )

        optimized_thickness_nm = np.zeros_like(
            beam_thickness_nm,
            dtype=np.float64,
        )

        optimized_thickness_nm[active_indices] = result.x
        num_iterations = result.nit

    else:
        optimized_thickness_nm = beam_thickness_nm.astype(np.float64)
        num_iterations = 0


    optimized_spectrum = calculate_transmittance_spectrum(
        fixed_eps,
        optimized_thickness_nm,
    )

    optimized_spectrum_mae = float(
        np.mean(np.abs(target_spectrum - optimized_spectrum))
    )


    """
    Thickness MAE：
    Variable-layer 結構需同時考慮正向結構與完整反向結構。
    """

    target_eps = target_params[0::2]
    target_thickness_nm = target_params[1::2]

    target_layer_count = int(np.sum(target_thickness_nm > 0.0))
    predicted_layer_count = int(np.sum(optimized_thickness_nm > 0.0))

    target_forward_thickness = target_thickness_nm[:target_layer_count]
    target_reverse_thickness = target_forward_thickness[::-1]

    target_forward_eps = target_eps[:target_layer_count]
    target_reverse_eps = target_forward_eps[::-1]

    predicted_active_thickness = optimized_thickness_nm[:predicted_layer_count]

    max_length = max(target_layer_count, predicted_layer_count)

    target_forward_padded = np.pad(
        target_forward_thickness,
        (0, max_length - target_layer_count),
    )

    target_reverse_padded = np.pad(
        target_reverse_thickness,
        (0, max_length - target_layer_count),
    )

    predicted_padded = np.pad(
        predicted_active_thickness,
        (0, max_length - predicted_layer_count),
    )

    forward_mae = np.mean(
        np.abs(predicted_padded - target_forward_padded)
    )

    reverse_mae = np.mean(
        np.abs(predicted_padded - target_reverse_padded)
    )


    if reverse_mae < forward_mae:
        aligned_target_thickness = np.pad(
            target_reverse_thickness,
            (0, 10 - target_layer_count),
        )

        aligned_target_eps = np.pad(
            target_reverse_eps,
            (0, 10 - target_layer_count),
        )

        alignment = "Inverted"
        optimized_thickness_mae = float(reverse_mae)

    else:
        aligned_target_thickness = np.pad(
            target_forward_thickness,
            (0, 10 - target_layer_count),
        )

        aligned_target_eps = np.pad(
            target_forward_eps,
            (0, 10 - target_layer_count),
        )

        alignment = "Normal"
        optimized_thickness_mae = float(forward_mae)


    optimized_params = np.zeros(20, dtype=np.float32)
    optimized_params[0::2] = fixed_eps
    optimized_params[1::2] = optimized_thickness_nm


    if plot:
        print(
            f"\nL-BFGS-B Refinement "
            f"({num_iterations} iterations) | "
            f"Alignment: {alignment}"
        )

        print("=" * 86)
        print(
            f"{'Layer':<8}"
            f"{'Material':<16}"
            f"{'Ground Truth':<20}"
            f"{'L-BFGS-B':<20}"
            f"{'Difference':<20}"
        )
        print("=" * 86)

        for i in range(10):
            target_d = aligned_target_thickness[i]
            optimized_d = optimized_thickness_nm[i]

            material = (
                fixed_eps[i]
                if optimized_d > 0
                else aligned_target_eps[i]
            )

            target_text = (
                f"{target_d:.2f} nm"
                if target_d > 0
                else "PAD"
            )

            optimized_text = (
                f"{optimized_d:.2f} nm"
                if optimized_d > 0
                else "PAD"
            )

            difference_text = (
                f"{optimized_d - target_d:+.2f} nm"
                if target_d > 0 and optimized_d > 0
                else "N/A"
            )

            print(
                f"{i + 1:<8}"
                f"{material:<16.1f}"
                f"{target_text:<20}"
                f"{optimized_text:<20}"
                f"{difference_text:<20}"
            )

        print("=" * 86)

        beam_spectrum = calculate_transmittance_spectrum(
            fixed_eps,
            beam_thickness_nm,
        )

        beam_spectrum_mae = float(
            np.mean(np.abs(target_spectrum - beam_spectrum))
        )

        plt.figure(figsize=(10, 5))

        plt.plot(
            WL_FULL,
            target_spectrum,
            label="Target",
            linewidth=2.5,
        )

        plt.plot(
            WL_FULL,
            beam_spectrum,
            label=f"Beam Search (MAE = {beam_spectrum_mae:.4f})",
            linestyle="--",
        )

        plt.plot(
            WL_FULL,
            optimized_spectrum,
            label=f"L-BFGS-B (MAE = {optimized_spectrum_mae:.4f})",
            linestyle=":",
        )

        plt.title(
            f"L-BFGS-B Refinement | "
            f"Thickness MAE = {optimized_thickness_mae:.2f} nm"
        )

        plt.xlabel("Wavelength (nm)")
        plt.ylabel("Transmittance")
        plt.ylim(-0.05, 1.05)
        plt.grid(True, linestyle=":")
        plt.legend()
        plt.tight_layout()
        plt.show()


    return (
        optimized_spectrum_mae,
        optimized_thickness_mae,
        optimized_params,
    )

In [ ]:
# ==============================================================================
# Block 5: Single Evaluation Function
# ==============================================================================

import numpy as np

# 實際載入訓練完成的模型
MODEL_PATH = "thin_film_transformer_variable_final.pth"
model, tokenizer, model_config = load_trained_model(MODEL_PATH)

def evaluate_single_variable_layers(target_params, target_spectrum, plot=True):
    """
    對單一 2~10 層結構執行完整反向設計評估：

    Beam Search
        ↓
    預測 Layer Count + Material + Discrete Thickness
        ↓
    L-BFGS-B Continuous Thickness Refinement

    Output:
        beam_spectrum_mae
        optimized_spectrum_mae
        optimized_thickness_mae
        target_layer_count
        predicted_layer_count
        layer_count_error
        beam_params
        optimized_params
    """

    # Beam Search
    beam_eps, beam_thickness_nm, beam_spectrum, beam_mae = run_beam_search_variable_layers(
        model,
        target_spectrum,
        beam_width=24,
    )

    beam_params = np.zeros(20, dtype=np.float32)
    beam_params[0::2] = beam_eps
    beam_params[1::2] = beam_thickness_nm

    # Layer Count
    target_layer_count = int(np.sum(target_params[1::2] > 0.0))
    predicted_layer_count = int(np.sum(beam_thickness_nm > 0.0))

    layer_count_error = int(
        target_layer_count != predicted_layer_count
    )

    # L-BFGS-B
    optimized_mae, optimized_thickness_mae, optimized_params = run_lbfgsb_variable_layers(
        target_params,
        target_spectrum,
        beam_params,
        max_iter=100,
        plot=plot,
    )

    if plot:
        print("\n" + "=" * 68)
        print(f"Ground Truth Layers      : {target_layer_count}")
        print(f"Predicted Layers         : {predicted_layer_count}")
        print(f"Layer Count Error        : {layer_count_error} (0 = Correct, 1 = Wrong)")
        print("-" * 68)
        print(f"Beam Search Spectrum MAE : {beam_mae:.6f}")
        print(f"L-BFGS-B Spectrum MAE    : {optimized_mae:.6f}")
        print(f"L-BFGS-B Thickness MAE   : {optimized_thickness_mae:.2f} nm")
        print("=" * 68)

    return {
        "beam_spectrum_mae": beam_mae,
        "optimized_spectrum_mae": optimized_mae,
        "optimized_thickness_mae": optimized_thickness_mae,
        "target_layer_count": target_layer_count,
        "predicted_layer_count": predicted_layer_count,
        "layer_count_error": layer_count_error,
        "beam_params": beam_params,
        "optimized_params": optimized_params,
    }

In [ ]:
# ==============================================================================
# Block 6: Single Test Run
# ==============================================================================

target_params, target_spectrum = generate_random_test_sample()

result = evaluate_single_variable_layers(
    target_params,
    target_spectrum,
    plot=True,
)

In [ ]:
# ==============================================================================
# Block 7: Batch Evaluation
# ==============================================================================

import numpy as np
import matplotlib.pyplot as plt


SUCCESS_THRESHOLD = 0.03


def run_batch_evaluation(num_runs=1000, plot_boxplot=True):
    """
    批量生成全新的 2~10 層結構並進行完整反向設計測試。

    統計：
        - Beam Search Spectrum MAE
        - L-BFGS-B Spectrum MAE
        - L-BFGS-B Thickness MAE
        - Layer Count Accuracy
        - Spectrum Success Rate

    Spectrum Success:
        optimized Spectrum MAE <= SUCCESS_THRESHOLD
    """

    beam_maes = []
    optimized_maes = []
    optimized_thickness_maes = []
    layer_errors = []
    target_layer_counts = []

    print(f"Running {num_runs} independent variable-layer test samples...")

    for i in range(num_runs):

        target_params, target_spectrum = generate_random_test_sample()

        result = evaluate_single_variable_layers(
            target_params,
            target_spectrum,
            plot=False,
        )

        beam_maes.append(result["beam_spectrum_mae"])
        optimized_maes.append(result["optimized_spectrum_mae"])
        optimized_thickness_maes.append(result["optimized_thickness_mae"])
        layer_errors.append(result["layer_count_error"])
        target_layer_counts.append(result["target_layer_count"])

        if (i + 1) % max(1, num_runs // 10) == 0:
            print(f"Progress: {i + 1}/{num_runs}")


    beam_maes = np.asarray(beam_maes)
    optimized_maes = np.asarray(optimized_maes)
    optimized_thickness_maes = np.asarray(optimized_thickness_maes)
    layer_errors = np.asarray(layer_errors)
    target_layer_counts = np.asarray(target_layer_counts)


    # 整體統計
    beam_mae_mean = np.mean(beam_maes)
    beam_mae_std = np.std(beam_maes)

    optimized_mae_mean = np.mean(optimized_maes)
    optimized_mae_std = np.std(optimized_maes)

    thickness_mae_mean = np.mean(optimized_thickness_maes)
    thickness_mae_std = np.std(optimized_thickness_maes)

    layer_accuracy = (1.0 - np.mean(layer_errors)) * 100.0

    success_mask = optimized_maes <= SUCCESS_THRESHOLD
    success_rate = np.mean(success_mask) * 100.0
    failure_rate = 100.0 - success_rate


    print("\n" + "=" * 70)
    print(f"{'Metric':<32}{'Beam Search':<18}{'L-BFGS-B':<18}")
    print("=" * 70)

    print(
        f"{'Spectrum MAE':<32}"
        f"{beam_mae_mean:.4f} ± {beam_mae_std:.4f}   "
        f"{optimized_mae_mean:.4f} ± {optimized_mae_std:.4f}"
    )

    print(
        f"{'Thickness MAE (nm)':<32}"
        f"{'N/A':<18}"
        f"{thickness_mae_mean:.2f} ± {thickness_mae_std:.2f}"
    )

    print(
        f"{'Layer Count Accuracy':<32}"
        f"{layer_accuracy:.2f}%"
    )

    print("=" * 70)

    print(f"\nSpectrum Success Threshold : MAE <= {SUCCESS_THRESHOLD:.3f}")
    print(f"Success Rate               : {success_rate:.2f}%")
    print(f"Failure Rate               : {failure_rate:.2f}%")
    print(f"Failures                    : {np.sum(~success_mask)}/{num_runs}")


    """
    各層數 Layer Count Accuracy
    """

    print("\nLayer Count Accuracy by Ground Truth Layers")
    print("-" * 42)

    for layer_count in range(2, 11):

        mask = target_layer_counts == layer_count

        if np.any(mask):
            accuracy = (1.0 - np.mean(layer_errors[mask])) * 100.0
            print(
                f"{layer_count:>2} Layers | "
                f"Accuracy: {accuracy:6.2f}% | "
                f"N = {np.sum(mask)}"
            )


    if plot_boxplot:

        plt.figure(figsize=(7, 4.5))

        plt.boxplot(
            [beam_maes, optimized_maes],
            tick_labels=["Beam Search", "L-BFGS-B"],
        )

        plt.axhline(
            SUCCESS_THRESHOLD,
            linestyle="--",
            label=f"Success Threshold = {SUCCESS_THRESHOLD:.2f}",
        )

        plt.title("Variable-Layer Spectrum MAE")
        plt.ylabel("Transmittance MAE")
        plt.grid(True, linestyle=":")
        plt.legend()
        plt.tight_layout()
        plt.show()


    return {
        "beam_maes": beam_maes,
        "optimized_maes": optimized_maes,
        "optimized_thickness_maes": optimized_thickness_maes,
        "layer_errors": layer_errors,
        "target_layer_counts": target_layer_counts,
        "layer_accuracy": layer_accuracy,
        "success_rate": success_rate,
        "failure_rate": failure_rate,
    }

In [ ]:
# ==============================================================================
# Block 8: Run Batch Test
# ==============================================================================

batch_result = run_batch_evaluation(
    num_runs=1000,
    plot_boxplot=True,
)